# 3.4 — ReAct Agent from Scratch

In notebooks 3.1–3.2 we used the tool-calling API (structured JSON tool calls).
Here we implement **ReAct** the original way — the model outputs **text** with a strict format:

```
Thought: I need to find the capital of France.
Action: get_capital
Action Input: France
Observation: Paris
Thought: I now know the answer.
Final Answer: The capital of France is Paris.
```

We parse this text ourselves — this is exactly what early LangChain agents did under the hood.

In [ ]:
!pip install langchain langchain-ollama --quiet

## Step 1 — Define Tools as Plain Functions

In [ ]:
import math

def calculator(expression: str) -> str:
    try:
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return str(result)
    except Exception as e:
        return f'Error: {e}'

def get_capital(country: str) -> str:
    capitals = {
        'france': 'Paris', 'germany': 'Berlin', 'japan': 'Tokyo',
        'india': 'New Delhi', 'usa': 'Washington D.C.', 'brazil': 'Brasilia',
        'australia': 'Canberra', 'canada': 'Ottawa',
    }
    return capitals.get(country.lower().strip(), f'Capital of {country} not found.')

def get_population(country: str) -> str:
    populations = {
        'france': '68 million', 'germany': '84 million', 'japan': '125 million',
        'india': '1.4 billion', 'usa': '335 million', 'brazil': '215 million',
    }
    return populations.get(country.lower().strip(), f'Population of {country} not found.')

# Tool registry — maps name → (function, description)
TOOLS = {
    'calculator':     (calculator,      'Evaluate a math expression. Input: the expression string.'),
    'get_capital':    (get_capital,     'Get the capital city of a country. Input: country name.'),
    'get_population': (get_population,  'Get the population of a country. Input: country name.'),
}

tools_description = '\n'.join(
    f'- {name}: {desc}' for name, (_, desc) in TOOLS.items()
)
print('Tools available:')
print(tools_description)

## Step 2 — The ReAct Prompt

The system prompt teaches the model the exact format to follow.

In [ ]:
REACT_SYSTEM_PROMPT = f"""You are a reasoning agent. Answer questions by thinking step by step.

You have access to these tools:
{tools_description}

Use this EXACT format:

Thought: reason about what to do
Action: tool_name
Action Input: the input to the tool
Observation: [tool result will be inserted here]
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer.
Final Answer: your final answer here

Rules:
- ALWAYS start with Thought:
- ALWAYS end with Final Answer:
- Only use tools listed above
- Do not make up observations — stop after Action Input and wait
"""
print('ReAct prompt set.')

## Step 3 — Parse the Model Output

In [ ]:
import re

def parse_react_output(text: str) -> dict:
    """
    Parse a ReAct-formatted LLM response.
    Returns: {'type': 'action', 'tool': ..., 'input': ...}
          or {'type': 'final',  'answer': ...}
    """
    # Check for Final Answer
    final_match = re.search(r'Final Answer:\s*(.+)', text, re.DOTALL)
    if final_match:
        return {'type': 'final', 'answer': final_match.group(1).strip()}

    # Check for Action
    action_match = re.search(r'Action:\s*(\w+)', text)
    input_match  = re.search(r'Action Input:\s*(.+)', text)

    if action_match and input_match:
        return {
            'type':  'action',
            'tool':  action_match.group(1).strip(),
            'input': input_match.group(1).strip(),
        }

    return {'type': 'unknown', 'raw': text}

# Test the parser
sample = """
Thought: I need to find the capital of France.
Action: get_capital
Action Input: France
"""
print(parse_react_output(sample))

sample2 = 'Final Answer: The capital is Paris.'
print(parse_react_output(sample2))

## Step 4 — The ReAct Loop

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOllama(model='llama3.1', temperature=0)

def react_agent(question: str, max_steps: int = 8) -> str:
    print(f'Question: {question}')
    print('-' * 50)

    # Build prompt: system + question
    prompt = f'{question}\n'
    messages = [
        SystemMessage(content=REACT_SYSTEM_PROMPT),
        HumanMessage(content=prompt)
    ]

    for step in range(max_steps):
        response = llm.invoke(messages)
        output_text = response.content

        # Print the model's reasoning
        for line in output_text.strip().split('\n'):
            if line.strip():
                print(f'  {line}')

        parsed = parse_react_output(output_text)

        if parsed['type'] == 'final':
            print()
            print(f'Final Answer: {parsed["answer"]}')
            return parsed['answer']

        if parsed['type'] == 'action':
            tool_name  = parsed['tool']
            tool_input = parsed['input']

            if tool_name not in TOOLS:
                observation = f'Error: tool "{tool_name}" does not exist.'
            else:
                tool_fn, _ = TOOLS[tool_name]
                observation = tool_fn(tool_input)

            print(f'  Observation: {observation}')

            # Append model output + observation to continue the loop
            messages.append(AIMessage(content=output_text))
            messages.append(HumanMessage(content=f'Observation: {observation}'))
        else:
            print('  [could not parse output — stopping]')
            break

    return 'Max steps reached without a final answer.'

print('ReAct agent ready.')

## Step 5 — Run the ReAct Agent

In [ ]:
react_agent('What is the capital of Japan?')

In [ ]:
react_agent('What is 123 multiplied by 456?')

In [ ]:
# Multi-step: requires two different tools
react_agent('What is the population of India, and what is half of that number?')

## Step 6 — ReAct (text) vs Tool Calling (JSON)

| Feature | ReAct (text format) | Tool Calling (JSON) |
|---------|--------------------|-----------------|
| How tools are called | Model outputs text with Action/Input format | Model outputs structured JSON |
| Parsing | You write a regex parser | LangChain handles it |
| Reliability | Can misformat | More reliable |
| Transparency | Full reasoning visible in text | Cleaner but less visible |
| Works with any LLM | Yes | Requires tool-calling support |

Modern LangChain uses the JSON tool calling API — but understanding the text-based ReAct loop helps you debug and customise agents.